ONLY LAST CELL IS GIVING GOOD RESULTS

In [ ]:
!pip install unstructured[local-inference]
!pip install pytesseract pdf2image pillow
!pip install langchain langchain-community
!pip install langchain-groq
!pip install chromadb
!pip install python-dotenv
!pip install tiktoken

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from unstructured.partition.pdf import partition_pdf

def parse_pdf(file_path):

    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",                 # layout detection
        infer_table_structure=True,
        languages=["eng"],             # tesseract OCR
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )

    return elements

In [3]:
import os

def load_documents(folder):

    all_elements = []

    for file in os.listdir(folder):

        if file.endswith(".pdf"):

            path = os.path.join(folder, file)

            print("Processing:", file)

            elements = parse_pdf(path)

            all_elements.extend(elements)

    print("Total elements:", len(all_elements))

    return all_elements

In [4]:
from unstructured.chunking.title import chunk_by_title

def create_chunks(elements):

    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )

    print("Chunks created:", len(chunks))

    return chunks

In [5]:
def chunk_to_text(chunk):

    if hasattr(chunk, "text"):
        return chunk.text

    return ""

In [6]:
from langchain_groq import ChatGroq
import json

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

def extract_structured_data(text):

    prompt = f"""
Extract structured information from this document.

Return JSON with fields:

personal_information
education
subjects
scores
skills
projects
certifications
employment_history

TEXT:
{text}
"""

    response = llm.invoke(prompt)

    return response.content

/workspaces/ai-research-platform/backend/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def build_structured_dataset(chunks):

    structured_records = []

    for i, chunk in enumerate(chunks):

        text = chunk_to_text(chunk)

        if len(text) < 50:
            continue

        print("Processing chunk", i)

        data = extract_structured_data(text)

        structured_records.append(data)

    return structured_records

In [8]:
def save_json(data, file="portfolio_data.json"):

    with open(file, "w") as f:
        json.dump(data, f, indent=2)

In [9]:
import os
import shutil

VECTOR_DB_PATH = "/workspaces/ai-research-platform/llm_lab/vector_db"

if os.path.exists(VECTOR_DB_PATH):
    shutil.rmtree(VECTOR_DB_PATH)

print("Old vector DB removed")

Old vector DB removed


In [10]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

VECTOR_DB_PATH = "/workspaces/ai-research-platform/llm_lab/vector_db"
def build_vector_db(chunks):

    docs = []

    for chunk in chunks:

        text = chunk_to_text(chunk)

        if len(text) < 50:
            continue

        docs.append(Document(page_content=text))

    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small"
    )

    db = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory=VECTOR_DB_PATH
    )

    return db

In [11]:
def query_rag(db, query):

    retriever = db.as_retriever(search_kwargs={"k":3})

    docs = retriever.invoke(query)

    return docs

In [12]:
def generate_answer(docs, query):

    context = ""

    for d in docs:
        context += d.page_content + "\n"

    prompt = f"""
Answer the question using the context.

Context:
{context}

Question:
{query}
"""

    response = llm.invoke(prompt)

    return response.content

In [13]:
import os

CACHE_DIR = "cache"
ELEMENTS_DIR = os.path.join(CACHE_DIR, "elements")
CHUNKS_DIR = os.path.join(CACHE_DIR, "chunks")

os.makedirs(ELEMENTS_DIR, exist_ok=True)
os.makedirs(CHUNKS_DIR, exist_ok=True)
os.makedirs("cache/elements", exist_ok=True)
os.makedirs("cache/chunks", exist_ok=True)
os.makedirs("cache/structured", exist_ok=True)
os.makedirs("vector_db", exist_ok=True)

print("Cache folders ready")

Cache folders ready


In [14]:
import json

def main():

    folder = "/workspaces/ai-research-platform/llm_lab/data"

    elements = load_documents(folder)

    chunks = create_chunks(elements)

    structured_data = build_structured_dataset(chunks)

    save_json(structured_data)

    db = build_vector_db(chunks)

    print("Vector DB created")

    query = "What are Srikar's academic qualifications?"

    docs = query_rag(db, query)

    answer = generate_answer(docs, query)

    print(answer)

    elements_data = []

    for e in elements:
        item = {
            "type": e.__class__.__name__,
            "text": getattr(e, "text", ""),
            "metadata": getattr(e, "metadata", None).__dict__ if getattr(e, "metadata", None) else {}
        }
        elements_data.append(item)

    with open("cache/elements/all_elements.json", "w") as f:
        json.dump(elements_data, f, indent=2)

    print("Saved elements → cache/elements/all_elements.json")


    chunks_text = []

    for c in chunks:
        if hasattr(c, "text"):
            chunks_text.append(c.text)

    with open("cache/chunks/all_chunks.json", "w") as f:
        json.dump(chunks_text, f, indent=2)

    print("Saved chunks → cache/chunks/all_chunks.json")

    with open("cache/structured/portfolio_structured.json", "w") as f:
        json.dump(structured_data, f, indent=2)

    print("Saved structured data → cache/structured/portfolio_structured.json")

if __name__ == "__main__":
    main()

Processing: SCAN_20260306_172642207.pdf


2026-03-06 21:58:59.134834647 [W:onnxruntime:Default, device_discovery.cc:131 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename ""5620e0c7-8062-4dce-aeb7-520c7ef76171"" dit not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 382.23it/s, Materializing param=model.query_position_embeddings.weight]                             


Processing: Transcript (2).pdf


Cannot set stroke color because expected 4 components but got [0]
Cannot set stroke color because expected 4 components but got [1]
Cannot set stroke color because expected 4 components but got [0]
Cannot set stroke color because expected 4 components but got [1]
Cannot set stroke color because expected 4 components but got [0]
Cannot set stroke color because expected 4 components but got [1]
Cannot set stroke color because expected 4 components but got [0]
Cannot set stroke color because expected 4 components but got [1]


Processing: SRIKAR_KURAGAYALA_AI_cv.pdf
Processing: 18WJ1A05H4.pdf
Processing: SCAN_20260306_173032732.pdf
Processing: SCAN_20260306_172848415.pdf
Processing: SCAN_20260306_172711773.pdf
Processing: SCAN_20260306_171008433.pdf
Processing: 18WJ1A05H4_btech.pdf
Processing: SCAN_20260306_172603684.pdf
Total elements: 652
Chunks created: 35
Processing chunk 0
Processing chunk 1
Processing chunk 2
Processing chunk 3
Processing chunk 4
Processing chunk 5
Processing chunk 6
Processing chunk 7
Processing chunk 8
Processing chunk 9
Processing chunk 10
Processing chunk 11
Processing chunk 12
Processing chunk 13
Processing chunk 14
Processing chunk 15
Processing chunk 16
Processing chunk 17
Processing chunk 18
Processing chunk 19
Processing chunk 20
Processing chunk 21
Processing chunk 22
Processing chunk 23
Processing chunk 24
Processing chunk 25
Processing chunk 26
Processing chunk 27
Processing chunk 28
Processing chunk 29
Processing chunk 30
Processing chunk 31
Processing chunk 32
Processing 

TypeError: Object of type CoordinatesMetadata is not JSON serializable

In [ ]:
import json

with open("cache/chunks/all_chunks.json") as f:
    chunks = json.load(f)

In [16]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

db = Chroma(
    persist_directory="vector_db",
    embedding_function=OpenAIEmbeddings(
        model="text-embedding-3-small"
    )
)

print("Vector DB loaded")

Vector DB loaded


In [18]:
query = "What are Srikar's academic qualifications how many marks he got in each classs?"

docs = query_rag(db, query)

answer = generate_answer(docs, query)

print(answer)

Based on the provided context, Srikar's academic qualifications are as follows:

1. **B.Tech (Computer Science and Engineering)** from Guru Nanak Institutions Technical Campus:
   - He passed the degree examination in July 2022.
   - He was placed in **FIRST CLASS WITH DISTINCTION**.
   - The exact marks are not mentioned, but according to the classification, a First Class with Distinction typically requires a high percentage (above 80%).

2. **Master of Science in Computing** from Dublin City University:
   - He was conferred the degree on 1 October 2025.
   - The exact marks are not mentioned in the provided context.

As for the marks in each class, the classification is as follows:

- **First Class Honours**: Not less than 70% (for Academic Year 2007/08 onwards)
- **Second Class Honours-Grade 1**: Not less than 63% (for Academic Year 2007/08 onwards)
- **Second Class Honours-Grade 2**: Not less than 55% (for Academic Year 2007/08 onwards)
- **Third Class Honours**: Not less than 50%

In [7]:
from unstructured.partition.pdf import partition_pdf
import pandas as pd

PDF_PATH = "data/Transcript (2).pdf"   # change this


def extract_rows(pdf_path):

    elements = partition_pdf(
        filename=pdf_path,
        strategy="ocr_only",
        languages=["eng"]
    )

    blocks = []

    for e in elements:

        if hasattr(e, "text") and e.text.strip():

            meta = e.metadata

            if hasattr(meta, "coordinates") and meta.coordinates:

                coords = meta.coordinates.points

                x = coords[0][0]
                y = coords[0][1]

                blocks.append({
                    "text": e.text.strip(),
                    "x": x,
                    "y": y,
                    "page": meta.page_number
                })

    # sort by page, vertical position, horizontal position
    blocks = sorted(blocks, key=lambda b: (b["page"], -b["y"], b["x"]))

    rows = []
    current_row = []
    current_y = None

    for block in blocks:

        if current_y is None:
            current_y = block["y"]

        # if y difference small → same row
        if abs(block["y"] - current_y) < 15:
            current_row.append(block["text"])
        else:
            rows.append(current_row)
            current_row = [block["text"]]
            current_y = block["y"]

    if current_row:
        rows.append(current_row)

    return rows


def print_table(rows):

    df = pd.DataFrame(rows)

    print("\nEXTRACTED TABLE:\n")
    print(df.to_string(index=False))


def main():

    rows = extract_rows(PDF_PATH)

    print_table(rows)


if __name__ == "__main__":
    main()

Cannot set stroke color because expected 4 components but got [0]
Cannot set stroke color because expected 4 components but got [1]
Cannot set stroke color because expected 4 components but got [0]
Cannot set stroke color because expected 4 components but got [1]



EXTRACTED TABLE:

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          0                                                                           1                                 2
                                                                                                                                                                    NB This is an important document of record of your examination results which you should keep carefully. Replacement copies of this Stat

In [8]:
from unstructured.partition.pdf import partition_pdf

def extract_pdf_text(pdf_path):

    elements = partition_pdf(
        filename=pdf_path,
        strategy="ocr_only",
        languages=["eng"]
    )

    text = ""

    for e in elements:
        if hasattr(e, "text"):
            text += e.text + "\n"

    return text

In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

def document_agent(document_text, question):

    prompt = f"""
You are a document analysis agent.

Rules:
- Answer ONLY using the document content.
- If information is not present, say "Not found in document".
- Extract numbers exactly as written.

DOCUMENT:
{document_text}

QUESTION:
{question}
"""

    response = llm.invoke(prompt)

    return response.content

In [11]:
PDF_PATH = "/workspaces/ai-research-platform/llm_lab/data/18WJ1A05H4_btech.pdf"

doc_text = extract_pdf_text(PDF_PATH)

question = "What is Srikar's CGPA?"

answer = document_agent(doc_text, question)

print(answer)

Not found in document.


In [13]:
from pdf2image import convert_from_path
import pytesseract

PDF_PATH = "/workspaces/ai-research-platform/llm_lab/data/18WJ1A05H4_btech.pdf"

def extract_text_from_scanned_pdf(pdf_path):
    images = convert_from_path(pdf_path)

    full_text = ""

    for i, img in enumerate(images):
        text = pytesseract.image_to_string(img)
        full_text += f"\n--- PAGE {i+1} ---\n{text}"

    return full_text


text = extract_text_from_scanned_pdf(PDF_PATH)

print(text[:3000])  # inspect extracted text


--- PAGE 1 ---
GURU NANAK INSTITUTIONS TECHNICAL CAMPUS

-AN UGC AUTONOMOUS INSTITUTION ~

a -
Approved by AICTE, New Delhi | Permanently Affiliated to NTU Hyderabad | Accredited by NAAC with A+ Grade Ke 2)
Campus: Ibrahimpatnam, R.R. District - 501 506. Telangana, India. 7 ( Li]
atone
—————— ay
PROVISIONAL CERTIFICATE —— “a
pene: WJ04623

31221WJ02651

21221WJ02651 Hall Ticket No. : 18WJ1A05H4

E
|

This is to certify that _Mr.KURAGAYALA SRIKAR

son of Mr. KURAGAYALA SRIHARI

passed B.TECH (COMPUTER SCIENCE AND ENGINEERING) |
degree examination of this Institute heldin July 2022 |
and that he was placed in FIRST CLASS WITH DISTINCTION.

He has satisfied all the requirements for the award of _B.TECH

degree of the Jawaharlal Nehru Technological University, Hyderabad .

ONAN

VERIFIED BY CONTROLLER OF EXAMINATIONS

mea GURU NANAK |
INSTITUTIONS

Els ae]


--- PAGE 2 ---
GURU NANAK INSTITUTIONS TECHNICAL CAMPUS

AN UGC AUTONOMOUS INSTITUTION

Approved by AICTE, New Delhi | Permanently A

In [14]:
answer = document_agent(text, "List all subjects with their marks.")
print(answer)

Not found in document.


In [15]:
!pip install docling

  Obtaining dependency information for docling from https://files.pythonhosted.org/packages/6b/08/0330d839d618b79188f8087551662dfbb8efd6fbfd5d33a2511138d98df0/docling-2.77.0-py3-none-any.whl.metadata
  Obtaining dependency information for docling-core[chunking]<3.0.0,>=2.66.0 from https://files.pythonhosted.org/packages/dc/66/d8bbe25dec2bb91d9090b939349b1c9b94c307edceada46c5bc6f213a569/docling_core-2.68.0-py3-none-any.whl.metadata
  Obtaining dependency information for docling-parse<6.0.0,>=5.3.2 from https://files.pythonhosted.org/packages/15/12/0d71fa0ce26b7f77ebd86de1108a4954a08be4969ff48fd027ed90a90594/docling_parse-5.5.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency information for docling-ibm-models<4,>=3.9.1 from https://files.pythonhosted.org/packages/ef/5d/97e9c2e10fbd3ee1723ac82c335f8211a9633c0397cc11ed057c3ba4006e/docling_ibm_models-3.11.0-py3-none-any.whl.metadata
  Obtaining dependency information for huggingface_hub<1,>=0.23 f

In [20]:
from docling.document_converter import DocumentConverter

pdf_path = "/workspaces/ai-research-platform/llm_lab/data/18WJ1A05H4_btech.pdf"

converter = DocumentConverter()
result = converter.convert(pdf_path)

document = result.document

for table in document.tables:
    df = table.export_to_dataframe()
    print("\nDetected Table:\n")
    print(df)

[INFO] 2026-03-07 18:01:38,833 [RapidOCR] base.py:22: Using engine_name: onnxruntime


[INFO] 2026-03-07 18:01:38,868 [RapidOCR] download_file.py:60: File exists and is valid: /workspaces/ai-research-platform/backend/venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-07 18:01:38,869 [RapidOCR] main.py:53: Using /workspaces/ai-research-platform/backend/venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-07 18:01:39,335 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-07 18:01:39,348 [RapidOCR] download_file.py:60: File exists and is valid: /workspaces/ai-research-platform/backend/venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-07 18:01:39,355 [RapidOCR] main.py:53: Using /workspaces/ai-research-platform/backend/venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-07 18:01:39,544 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-07 18:01:39,600 [RapidOCR] 

ImportError: cannot import name 'is_flax_available' from 'transformers.utils' (/workspaces/ai-research-platform/backend/venv/lib/python3.12/site-packages/transformers/utils/__init__.py)

In [18]:
!pip uninstall transformers -y
!pip install transformers==4.41.2

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
  Obtaining dependency information for transformers==4.41.2 from https://files.pythonhosted.org/packages/d9/b7/98f821d70102e2d38483bbb7013a689d2d646daa4495377bc910374ad727/transformers-4.41.2-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 620.2 kB/s eta 0:00:00 0:00:01
  Obtaining dependency information for tokenizers<0.20,>=0.19 from https://files.pythonhosted.org/packages/80/54/12047a69f5b382d7ee72044dc89151a2dd0d13b2c9bdcc22654883704d31/tokenizers-0.19.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.6 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.0 MB/s eta 0:00:00:00:01
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Succ

In [19]:
!pip install --upgrade docling

  Obtaining dependency information for transformers<5.0.0,>=4.34.0 from https://files.pythonhosted.org/packages/03/b8/e484ef633af3887baeeb4b6ad12743363af7cce68ae51e938e00aaa0529d/transformers-4.57.6-py3-none-any.whl.metadata
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Obtaining dependency information for tokenizers<=0.23.0,>=0.22.0 from https://files.pythonhosted.org/packages/2e/76/932be4b50ef6ccedf9d3c6639b056a967a86258c6d9200643f01269211ca/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attem

In [7]:
import os
import re
import json
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import fitz  # PyMuPDF
from pdf2image import convert_from_path
import pytesseract
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# =========================
# CONFIG
# =========================

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")

# Optional vision hook support.
# Set to True only if you wire in a vision-capable model yourself.
ENABLE_VISION_FALLBACK = False

CACHE_DIR = Path("doc_cache")
CACHE_DIR.mkdir(exist_ok=True)

POPPLER_PATH = None  # Optional on Windows
TESSERACT_CMD = None  # Optional on Windows, e.g. r"C:\Program Files\Tesseract-OCR\tesseract.exe"

if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

# =========================
# HELPERS
# =========================

def file_sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()


def cache_path(pdf_path: str, suffix: str) -> Path:
    file_hash = file_sha256(pdf_path)
    stem = Path(pdf_path).stem
    return CACHE_DIR / f"{stem}_{file_hash[:12]}_{suffix}.json"


def save_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def text_quality_score(text: str) -> float:
    """
    Heuristic score for extraction quality.
    Higher is better.
    """
    if not text or len(text.strip()) < 20:
        return 0.0

    total_chars = len(text)
    alnum_chars = sum(ch.isalnum() for ch in text)
    digit_count = sum(ch.isdigit() for ch in text)

    weird_ratio = 1 - (alnum_chars / max(total_chars, 1))
    line_count = len(text.splitlines())
    useful_lines = sum(1 for line in text.splitlines() if len(line.strip()) > 3)

    keywords = [
        "hall ticket", "name", "branch", "semester", "cgpa", "credits",
        "subject", "grade", "marks", "examination", "university", "institute"
    ]
    keyword_hits = sum(1 for kw in keywords if kw.lower() in text.lower())

    score = 0.0
    score += min(len(text) / 5000, 1.0) * 0.25
    score += min(useful_lines / max(line_count, 1), 1.0) * 0.20
    score += min(keyword_hits / 6, 1.0) * 0.30
    score += min(digit_count / 50, 1.0) * 0.15
    score += max(0.0, 1 - weird_ratio) * 0.10
    return round(score, 4)


def looks_like_scanned_pdf(pdf_path: str) -> bool:
    """
    Detect whether the PDF is likely scanned by checking embedded text.
    """
    doc = fitz.open(pdf_path)
    text = []
    for page in doc:
        text.append(page.get_text("text"))
    combined = clean_text("\n".join(text))
    return len(combined) < 100


# =========================
# EXTRACTION PIPELINES
# =========================

def extract_direct_text(pdf_path: str) -> Dict[str, Any]:
    """
    Fast path for digital PDFs with embedded text.
    """
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc, start=1):
        txt = page.get_text("text")
        pages.append({
            "page": i,
            "text": clean_text(txt)
        })

    combined = clean_text("\n\n".join(p["text"] for p in pages))
    return {
        "pipeline": "direct_text",
        "pages": pages,
        "text": combined,
        "score": text_quality_score(combined)
    }


def extract_ocr_text(pdf_path: str) -> Dict[str, Any]:
    """
    OCR fallback using pdf2image + pytesseract.
    """
    images = convert_from_path(pdf_path, poppler_path=POPPLER_PATH)
    pages = []

    for i, img in enumerate(images, start=1):
        txt = pytesseract.image_to_string(img, lang="eng")
        pages.append({
            "page": i,
            "text": clean_text(txt)
        })

    combined = clean_text("\n\n".join(p["text"] for p in pages))
    return {
        "pipeline": "ocr_tesseract",
        "pages": pages,
        "text": combined,
        "score": text_quality_score(combined)
    }


def extract_vision_fallback(pdf_path: str) -> Optional[Dict[str, Any]]:
    """
    Placeholder for a vision-capable model.
    Wire this up if you have a provider that accepts page images.
    Return the same structure as the other extractors.

    Example future flow:
    - convert PDF pages to images
    - send images to a vision model
    - ask it to transcribe / structure content
    """
    if not ENABLE_VISION_FALLBACK:
        return None

    # Implement if you have a vision endpoint/model.
    return None


# =========================
# CHOOSE BEST EXTRACTION
# =========================

def run_extraction_pipelines(pdf_path: str, force_refresh: bool = False) -> Dict[str, Any]:
    cache = cache_path(pdf_path, "best_extraction")
    if cache.exists() and not force_refresh:
        return load_json(cache)

    results: List[Dict[str, Any]] = []

    # Always try direct extraction first.
    direct = extract_direct_text(pdf_path)
    results.append(direct)

    # OCR if scanned OR direct extraction quality is weak.
    if looks_like_scanned_pdf(pdf_path) or direct["score"] < 0.45:
        ocr = extract_ocr_text(pdf_path)
        results.append(ocr)

    vision = extract_vision_fallback(pdf_path)
    if vision:
        results.append(vision)

    best = max(results, key=lambda x: x["score"])
    final = {
        "pdf_path": pdf_path,
        "is_scanned_guess": looks_like_scanned_pdf(pdf_path),
        "candidates": [
            {"pipeline": r["pipeline"], "score": r["score"], "text_length": len(r["text"])}
            for r in results
        ],
        "best_pipeline": best["pipeline"],
        "best_score": best["score"],
        "pages": best["pages"],
        "text": best["text"]
    }

    save_json(cache, final)
    return final


# =========================
# LLM STRUCTURED EXTRACTION
# =========================

def get_llm() -> ChatGroq:
    if not GROQ_API_KEY:
        raise ValueError("Missing GROQ_API_KEY in environment.")
    return ChatGroq(model=GROQ_MODEL, temperature=0)


def chunk_text(text: str, chunk_size: int = 12000, overlap: int = 500) -> List[str]:
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + chunk_size, n)
        chunks.append(text[start:end])
        if end == n:
            break
        start = max(0, end - overlap)

    return chunks


def extract_structured_chunk(llm: ChatGroq, chunk_text_value: str) -> Dict[str, Any]:
    prompt = f"""
You are a strict document extraction engine.

Extract only what is explicitly present in the text below.
Do not guess. Do not infer missing numbers.
If a field is missing, use null or [].

Return ONLY valid JSON with this schema:

{{
  "personal_information": {{
    "name": null,
    "hall_ticket_number": null,
    "branch": null,
    "institution": null,
    "university": null,
    "year_of_admission": null,
    "month_year_of_pass": null,
    "classification": null,
    "cgpa": null,
    "credits_registered_secured": null
  }},
  "education_records": [
    {{
      "degree": null,
      "specialization": null,
      "institution": null,
      "university": null,
      "date_or_year": null,
      "classification": null,
      "cgpa": null
    }}
  ],
  "subjects": [
    {{
      "semester": null,
      "subject": null,
      "marks": null,
      "grade": null,
      "grade_points": null,
      "credits": null
    }}
  ],
  "skills": [],
  "projects": [],
  "certifications": [],
  "employment_history": []
}}

Text:
{chunk_text_value}
"""
    response = llm.invoke(prompt)
    content = response.content.strip()

    # Strip accidental markdown fences
    content = re.sub(r"^```json\s*", "", content)
    content = re.sub(r"^```\s*", "", content)
    content = re.sub(r"\s*```$", "", content)

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return {
            "personal_information": {},
            "education_records": [],
            "subjects": [],
            "skills": [],
            "projects": [],
            "certifications": [],
            "employment_history": [],
            "_raw_failed_output": content
        }


def merge_structured_results(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    merged = {
        "personal_information": {},
        "education_records": [],
        "subjects": [],
        "skills": [],
        "projects": [],
        "certifications": [],
        "employment_history": []
    }

    # Merge personal information by first non-null value.
    for result in results:
        pi = result.get("personal_information", {})
        for k, v in pi.items():
            if v not in (None, "", [], {}) and not merged["personal_information"].get(k):
                merged["personal_information"][k] = v

    # Append lists.
    list_keys = [
        "education_records", "subjects", "skills",
        "projects", "certifications", "employment_history"
    ]
    for key in list_keys:
        seen = set()
        merged_list = []
        for result in results:
            for item in result.get(key, []) or []:
                signature = json.dumps(item, sort_keys=True, ensure_ascii=False)
                if signature not in seen:
                    seen.add(signature)
                    merged_list.append(item)
        merged[key] = merged_list

    return merged


def llm_structured_extraction(pdf_path: str, best_text: str, force_refresh: bool = False) -> Dict[str, Any]:
    cache = cache_path(pdf_path, "structured")
    if cache.exists() and not force_refresh:
        return load_json(cache)

    llm = get_llm()
    chunks = chunk_text(best_text)
    extracted_chunks = []

    for idx, ch in enumerate(chunks, start=1):
        print(f"LLM extracting chunk {idx}/{len(chunks)} ...")
        extracted = extract_structured_chunk(llm, ch)
        extracted_chunks.append(extracted)

    merged = merge_structured_results(extracted_chunks)
    save_json(cache, merged)
    return merged


# =========================
# QA AGENT OVER EXTRACTED PDF TEXT
# =========================

def ask_document_question(pdf_path: str, question: str, force_refresh: bool = False) -> str:
    extraction = run_extraction_pipelines(pdf_path, force_refresh=force_refresh)
    llm = get_llm()

    prompt = f"""
You are a document QA agent.

Rules:
- Answer ONLY from the document text below.
- If not found, say: Not found in document.
- If asked for marks/subjects, return exact values only if explicitly present.
- Keep the answer concise.

Document text:
{extraction["text"]}

Question:
{question}
"""
    response = llm.invoke(prompt)
    return response.content


# =========================
# MAIN PIPELINE
# =========================

def process_pdf(pdf_path: str, force_refresh: bool = False) -> Dict[str, Any]:
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    print(f"\nProcessing: {pdf_path}")
    extraction = run_extraction_pipelines(pdf_path, force_refresh=force_refresh)

    print("\nCandidate extraction scores:")
    for c in extraction["candidates"]:
        print(f"  - {c['pipeline']}: score={c['score']}, text_length={c['text_length']}")

    print(f"\nSelected pipeline: {extraction['best_pipeline']} (score={extraction['best_score']})")

    structured = llm_structured_extraction(
        pdf_path=pdf_path,
        best_text=extraction["text"],
        force_refresh=force_refresh
    )

    output = {
        "pdf_path": pdf_path,
        "best_pipeline": extraction["best_pipeline"],
        "best_score": extraction["best_score"],
        "structured_data": structured
    }

    final_cache = cache_path(pdf_path, "final_output")
    save_json(final_cache, output)

    print(f"\nSaved final output to: {final_cache}")
    return output


# =========================
# BATCH PROCESS FOLDER
# =========================

def process_folder(folder_path: str, force_refresh: bool = False) -> List[Dict[str, Any]]:
    outputs = []
    for p in Path(folder_path).glob("*.pdf"):
        try:
            out = process_pdf(str(p), force_refresh=force_refresh)
            outputs.append(out)
        except Exception as e:
            print(f"Failed for {p.name}: {e}")
    return outputs


# =========================
# EXAMPLE USAGE
# =========================

if __name__ == "__main__":
    # Example 1: process one PDF
    pdf_path = "/workspaces/ai-research-platform/llm_lab/data/18WJ1A05H4_btech.pdf"
    result = process_pdf(pdf_path, force_refresh=False)

    print("\n=== STRUCTURED OUTPUT ===")
    print(json.dumps(result["structured_data"], indent=2, ensure_ascii=False))

    # Example 2: ask a question against the selected extraction text
    answer = ask_document_question(
        pdf_path,
        "What are Srikar's academic qualifications and CGPA?",
        force_refresh=False
    )
    print("\n=== QA ANSWER ===")
    print(answer)

    # Example 3: process a whole folder
    # all_results = process_folder("/workspaces/ai-research-platform/llm_lab/data", force_refresh=False)

/workspaces/ai-research-platform/backend/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing: /workspaces/ai-research-platform/llm_lab/data/18WJ1A05H4_btech.pdf

Candidate extraction scores:
  - direct_text: score=0.0, text_length=0
  - ocr_tesseract: score=0.8275, text_length=3405

Selected pipeline: ocr_tesseract (score=0.8275)
LLM extracting chunk 1/1 ...

Saved final output to: doc_cache/18WJ1A05H4_btech_8360ab5ba304_final_output.json

=== STRUCTURED OUTPUT ===
{
  "personal_information": {},
  "education_records": [],
  "subjects": [],
  "skills": [],
  "projects": [],
  "certifications": [],
  "employment_history": []
}

=== QA ANSWER ===
Srikar passed B.TECH (COMPUTER SCIENCE AND ENGINEERING) degree examination with FIRST CLASS WITH DISTINCTION. His Aggregate CGPA secured is 8.25.


In [ ]:
import base64
import json
from pdf2image import convert_from_path
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

PDF_PATH = "your_pdf_here.pdf"


def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def pdf_to_images(pdf_path):
    images = convert_from_path(pdf_path)
    paths = []

    for i, img in enumerate(images):
        path = f"page_{i}.png"
        img.save(path, "PNG")
        paths.append(path)

    return paths


def extract_with_vision(pdf_path):

    image_paths = pdf_to_images(pdf_path)

    images_content = []

    for p in image_paths:
        images_content.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:image/png;base64,{encode_image(p)}"
            }
        })

    prompt = """
Extract all academic information from this transcript.

Return structured JSON.

Example format:

{
 "name": "",
 "hall_ticket": "",
 "branch": "",
 "cgpa": "",
 "subjects":[
   {"subject":"", "marks":"", "grade":""}
 ]
}

Only return JSON.
Do not add explanations.
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "user",
                "content": [{"type": "text", "text": prompt}] + images_content
            }
        ],
        temperature=0
    )

    result = response.choices[0].message.content

    print("\n===== RAW LLM OUTPUT =====\n")
    print(result)

    try:
        data = json.loads(result)
        print("\n===== STRUCTURED JSON =====\n")
        print(json.dumps(data, indent=2))
    except:
        print("\nCould not parse JSON")


extract_with_vision(PDF_PATH)

In [13]:
import base64
import json
from pdf2image import convert_from_path
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

PDF_PATH = "/workspaces/ai-research-platform/llm_lab/data/18WJ1A05H4_btech.pdf"


# Convert image to base64
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()


# Convert PDF → images
def pdf_to_images(pdf_path):

    images = convert_from_path(pdf_path)

    paths = []

    for i, img in enumerate(images):
        p = f"page_{i}.png"
        img.save(p)
        paths.append(p)

    return paths


# Step 1 — Vision extraction
def extract_document_content(pdf_path):

    images = convert_from_path(pdf_path)

    all_text = []

    for i, img in enumerate(images):

        img_path = f"temp_page_{i}.jpg"
        img.thumbnail((2000, 2000))
        img.save(img_path, "JPEG", quality=80)

        with open(img_path, "rb") as f:
            base64_image = base64.b64encode(f.read()).decode()

        print(f"Processing page {i+1}/{len(images)}")

        response = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Extract ALL information visible in this document page."
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            temperature=0
        )

        page_text = response.choices[0].message.content
        all_text.append(page_text)

    return "\n".join(all_text)


# Step 2 — Structure data into JSON
def structure_to_json(extracted_text):

    struct_prompt = f"""
Convert the following document information into structured JSON.

Rules:
- Capture all entities and data fields present.
- Detect logical sections automatically.
- Create nested structures when appropriate.
- Do not invent information.
- Only use information found in the text.

Text:

{extracted_text}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": struct_prompt}
        ],
        temperature=0
    )

    result = response.choices[0].message.content

    try:
        return json.loads(result)
    except:
        print("Raw output:\n", result)
        return None



In [17]:
from pathlib import Path
import json
import re
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

DOC_FOLDER = "/workspaces/ai-research-platform/llm_lab/data"
OUTPUT_FOLDER = "/workspaces/ai-research-platform/llm_lab/results"
MAX_WORKERS = 2

Path(OUTPUT_FOLDER).mkdir(exist_ok=True)


def clean_llm_json(response_text):
    """
    Extract JSON from LLM responses even if wrapped in text or markdown
    """

    if not text:
        return None

    # Find JSON block
    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        print("⚠ No JSON found")
        return None

    json_text = match.group(0)

    try:
        return json.loads(json_text)

    except Exception as e:
        print("⚠ JSON parsing failed:", e)
        print("Extracted JSON text:", json_text[:500])
        return None


def process_single_pdf(pdf_path):

    pdf_path = Path(pdf_path)

    try:
        print(f"Processing: {pdf_path.name}")

        # Step 1 — extract raw text using vision model
        extracted_text = extract_document_content(str(pdf_path))

        # Step 2 — send to LLM for structuring
        raw_structured = structure_to_json(extracted_text)

        # Step 3 — clean and parse JSON
        structured = clean_llm_json(raw_structured)

        if structured is None:
            print(f"⚠ Failed to parse JSON for {pdf_path.name}")
            return f"Skipped {pdf_path.name}"

        # Step 4 — save JSON
        output_file = Path(OUTPUT_FOLDER) / f"{pdf_path.stem}.json"

        with open(output_file, "w") as f:
            json.dump(structured, f, indent=2)

        time.sleep(60)  # avoid Groq rate limits

        return f"Finished {pdf_path.name}"

    except Exception as e:
        return f"Failed {pdf_path.name}: {str(e)}"


def process_folder_parallel(folder):

    pdf_files = list(Path(folder).glob("*.pdf"))

    print(f"\nFound {len(pdf_files)} PDFs\n")

    results = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:

        futures = [executor.submit(process_single_pdf, str(p)) for p in pdf_files]

        for future in as_completed(futures):
            result = future.result()
            print(result)
            results.append(result)

    return results


if __name__ == "__main__":

    process_folder_parallel(DOC_FOLDER)


Found 10 PDFs



Processing: SCAN_20260306_172642207.pdfProcessing: Transcript (2).pdf

Processing page 1/1
Processing page 1/2
Raw output:
 ```json
{
  "document": {
    "type": "Provisional Certificate",
    "institution": {
      "name": "Guru Nanak Institutions Technical Campus",
      "approved_by": "AICTE, New Delhi",
      "affiliation": "Permanently Affiliated to JNTU Hyderabad",
      "accreditation": "Accredited by NAAC with A+ Grade",
      "location": "Ibrahimpatnam, R.R. District - 501 506, Telangana, India"
    },
    "certificate_details": {
      "certificate_number": "WJ04623",
      "hall_ticket_number": "18WJ1A05H4"
    },
    "student": {
      "name": "Mr. Kuragayala Srikar",
      "parent_name": "Mr. Kuragayala Srihari"
    },
    "academic": {
      "degree": "B.Tech (Computer Science and Engineering)",
      "examination_date": "July 2022",
      "result": "First Class with Distinction",
      "awarding_university": "Jawaharlal Nehru Technological University, Hyderabad"
    },
 

In [ ]:
import base64
import os
import time
import shutil
import tempfile
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

from pdf2image import convert_from_path
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

DOC_FOLDER = "/workspaces/ai-research-platform/llm_lab/data"
OUTPUT_FOLDER = "/workspaces/ai-research-platform/llm_lab/results"

MAX_WORKERS = 2

Path(OUTPUT_FOLDER).mkdir(exist_ok=True)


# ---------------------------
# Vision extraction
# ---------------------------
def extract_document_content(pdf_path):

    client = Groq(api_key=os.getenv("GROQ_API_KEY"))

    images = convert_from_path(pdf_path)

    all_text = []

    # create temp folder for this PDF
    temp_dir = tempfile.mkdtemp(prefix="pdf_pages_")

    try:

        for i, img in enumerate(images):

            img.thumbnail((2000, 2000))

            img_path = os.path.join(temp_dir, f"page_{i}.jpg")

            img.save(img_path, "JPEG", quality=80)

            with open(img_path, "rb") as f:
                base64_image = base64.b64encode(f.read()).decode()

            print(f"Processing page {i+1}/{len(images)}")

            response = client.chat.completions.create(
                model="meta-llama/llama-4-scout-17b-16e-instruct",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": "Extract ALL information visible in this document page."
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_image}"
                                }
                            }
                        ]
                    }
                ],
                temperature=0
            )

            page_text = response.choices[0].message.content

            all_text.append(page_text)

    finally:
        # delete temp folder and all images
        shutil.rmtree(temp_dir, ignore_errors=True)

    return "\n".join(all_text)


# ---------------------------
# Structure document text
# ---------------------------
def structure_document_text(extracted_text):

    client = Groq(api_key=os.getenv("GROQ_API_KEY"))

    struct_prompt = f"""
Organize the following document information clearly.

Use headings, sections, and structured formatting.
Do NOT invent information.
Only use information present in the text.

Document content:

{extracted_text}
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": struct_prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content


# ---------------------------
# Process a single PDF
# ---------------------------
def process_single_pdf(pdf_path):

    pdf_path = Path(pdf_path)

    try:

        print(f"\nProcessing: {pdf_path.name}")

        extracted_text = extract_document_content(str(pdf_path))

        structured_text = structure_document_text(extracted_text)

        output_file = Path(OUTPUT_FOLDER) / f"{pdf_path.stem}.txt"

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(structured_text)

        time.sleep(60)  # avoid Groq rate limits

        return f"Finished {pdf_path.name}"

    except Exception as e:

        return f"Failed {pdf_path.name}: {str(e)}"


# ---------------------------
# Process folder
# ---------------------------
def process_folder_parallel(folder):

    pdf_files = list(Path(folder).glob("*.pdf"))

    print(f"\nFound {len(pdf_files)} PDFs\n")

    results = []

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:

        futures = [executor.submit(process_single_pdf, str(p)) for p in pdf_files]

        for future in as_completed(futures):

            result = future.result()

            print(result)

            results.append(result)

    return results


# ---------------------------
# Main
# ---------------------------
if __name__ == "__main__":

    process_folder_parallel(DOC_FOLDER)


Found 1 PDFs


Processing: SCAN_20260306_172848415.pdf
Processing page 1/5
Processing page 2/5
Processing page 3/5
Processing page 4/5
Processing page 5/5
Finished SCAN_20260306_172848415.pdf
